# DSN Mart — product-history investigation

This notebook checks whether sales of the same product in other stores explain errors left by
the price/store model. Product histories and preprocessing are learned **only from each training
fold**. No validation sales, test labels, external sales data, or row-number patterns are used.

It compares the existing spline model, three shrunk product-residual corrections, and CatBoost
with product identity explicitly included. Model selection uses the original development folds.
The previous holdout is only a diagnostic. An additional seed checks stability but reuses development
rows and is not an independent test. We do not claim these checks prove an irreducible error floor.

Run beside the supplied CSVs with the existing project environment. This follow-up loads the trusted
locally saved pipelines in `outputs/` and `outputs/improvement/`. It prints progress and saves results
under `outputs/product_history/`. RMSE remains in original sales units; 50 is a target, not a promise.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import cloudpickle
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error
from IPython.display import display
from threadpoolctl import threadpool_limits

OUT = Path('outputs/product_history')
OUT.mkdir(parents=True, exist_ok=True)
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')
with open('outputs/improvement/model_bundle.pkl', 'rb') as f:
    previous = cloudpickle.load(f)
with open('outputs/retail_sales_pipeline.pkl', 'rb') as f:
    catboost_reference = cloudpickle.load(f)
features = previous['features']
baseline = clone(previous['models'][0])
X, y = train[features], train.total_sales
Xd, Xh, yd, yh = train_test_split(X, y, test_size=.2, random_state=42)
assert not train.duplicated(['product_code', 'store_code']).any()
counts = Xd.product_code.value_counts()
print('Product observations per development product:')
display(counts.describe())
print('Test rows with a product observed in training:', test.product_code.isin(train.product_code).sum())


## 1. Models that use product history
The residual model first fits the store-price spline, then estimates a product-specific correction
from its training residuals. Corrections are shrunk toward zero: sum(residuals)/(count + shrinkage).
Unseen products get zero correction. Residuals are in-sample within the training fold; that can
attenuate or overfit corrections, which is why performance is measured only on held-out rows.
No validation target is included in these statistics.

CatBoost receives product identity as a category. Its categorical processing is learned inside
each training fit. We do not pass the validation targets as features or use them for early stopping.


In [ ]:
class ProductResidual(RegressorMixin, BaseEstimator):
    def __init__(self, base, shrinkage=20.):
        self.base = base
        self.shrinkage = shrinkage

    def fit(self, X, y):
        self.base_ = clone(self.base).fit(X, y)
        residual = np.asarray(y) - self.base_.predict(X)
        history = pd.DataFrame({'product': X.product_code.to_numpy(), 'residual': residual})
        stats = history.groupby('product').residual.agg(['sum', 'count'])
        self.correction_ = stats['sum'] / (stats['count'] + self.shrinkage)
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        return self.base_.predict(X) + X.product_code.map(self.correction_).fillna(0).to_numpy()

class KeepProductIdentity(TransformerMixin, BaseEstimator):
    def __init__(self, base):
        self.base = base

    def fit(self, X, y=None):
        self.base_ = clone(self.base).fit(X, y)
        return self

    def transform(self, X):
        result = self.base_.transform(X)
        result['product_identity'] = X.product_code.fillna('unknown').astype(str).astype(object)
        return result

cat = clone(catboost_reference.named_steps['model'])
cat.set_params(iterations=600, depth=4, l2_leaf_reg=20, verbose=200,
    cat_features=tuple(cat.get_params()['cat_features']) + ('product_identity',))
candidates = {'Previous spline': clone(baseline)}
for shrinkage in [5., 20., 100.]:
    candidates[f'Product residual shrink {shrinkage:g}'] = ProductResidual(baseline, shrinkage)
candidates['CatBoost with product identity'] = Pipeline([
    ('features', KeepProductIdentity(catboost_reference.named_steps['features'])), ('model', cat)])


## 2. Compare identical held-out folds
The baseline must reproduce the previous development RMSE. Candidate choices are fixed above.
The target of 50 is checked against actual original-unit error, never rescaled error.


In [ ]:
scores, oof, records = {}, {}, []
with threadpool_limits(limits=4):
    for name, template in candidates.items():
        pred = np.zeros(len(Xd))
        for fold, (a, b) in enumerate(KFold(5, shuffle=True, random_state=42).split(Xd), 1):
            print(f'START {name} fold {fold}/5', flush=True)
            fitted = clone(template).fit(Xd.iloc[a], yd.iloc[a])
            pred[b] = np.maximum(0, fitted.predict(Xd.iloc[b]))
            rmse = float(root_mean_squared_error(yd.iloc[b], pred[b]))
            records.append({'model': name, 'fold': fold, 'rmse': rmse})
            pd.DataFrame(records).to_csv(OUT / 'fold_scores.csv', index=False)
            print(f'DONE: validation RMSE {rmse:.3f}', flush=True)
        oof[name] = pred
        scores[name] = float(root_mean_squared_error(yd, pred))
        print(f'{name}: OOF RMSE {scores[name]:.3f}', flush=True)
comparison = pd.Series(scores, name='rmse').sort_values()
comparison.to_csv(OUT / 'comparison.csv', index_label='model')
pd.DataFrame(oof, index=Xd.index).assign(actual=yd).to_csv(OUT / 'oof.csv', index_label='training_row')
assert abs(scores['Previous spline'] - 1073.6421072713474) < .01
winner = comparison.index[0]
display(comparison)


## 3. Stability and previously inspected holdout diagnostics
The winner is fixed before this section. This additional split cannot turn model-selection
results into independent evidence. If it contradicts the main result, that uncertainty is reported.


In [ ]:
stability = {}
with threadpool_limits(limits=4):
    for name in dict.fromkeys(['Previous spline', winner]):
        pred = np.zeros(len(Xd))
        for fold, (a, b) in enumerate(KFold(5, shuffle=True, random_state=93).split(Xd), 1):
            print(f'Stability: {name}, fold {fold}/5', flush=True)
            fitted = clone(candidates[name]).fit(Xd.iloc[a], yd.iloc[a])
            pred[b] = np.maximum(0, fitted.predict(Xd.iloc[b]))
        stability[name] = float(root_mean_squared_error(yd, pred))
    selected = clone(candidates[winner]).fit(Xd, yd)
    holdout_rmse = float(root_mean_squared_error(yh, np.maximum(0, selected.predict(Xh))))
improved = scores[winner] < scores['Previous spline'] and stability[winner] < stability['Previous spline']
report = {'selected': winner, 'cv_rmse': scores[winner],
    'previous_cv_rmse': scores['Previous spline'], 'second_seed_rmse': stability[winner],
    'previous_second_seed_rmse': stability['Previous spline'],
    'previously_inspected_holdout_rmse': holdout_rmse,
    'target_rmse': 50, 'target_met_on_both_cv_checks': bool(max(scores[winner], stability[winner]) <= 50),
    'new_submission_created': bool(improved),
    'note': 'Model-selection and reused-data diagnostics; no claimed leaderboard improvement or proven error floor.'}
display(pd.Series(report))


## 4. Save only a supported new candidate, with a submission description
Export requires improvement over the reference on both development split seeds. All older
submissions stay intact. The measured holdout diagnostic is reported even if it is worse.


In [ ]:
if improved:
    with threadpool_limits(limits=4):
        final = clone(candidates[winner]).fit(X, y)
        predictions = np.maximum(0, final.predict(test[features]))
    submission = sample[['id']].copy()
    submission['total_sales'] = submission.id.map(pd.Series(predictions, index=test.id))
    assert submission.shape == sample.shape and submission.id.equals(sample.id)
    assert submission.id.is_unique and np.isfinite(submission.total_sales).all()
    assert submission.total_sales.ge(0).all()
    submission.to_csv(OUT / 'submission_product_history.csv', index=False)
    with (OUT / 'pipeline.pkl').open('wb') as f:
        cloudpickle.dump(final, f)
    with (OUT / 'pipeline.pkl').open('rb') as f:
        restored = cloudpickle.load(f)
    assert np.allclose(predictions, np.maximum(0, restored.predict(test[features])))
    description = (f'{winner}, using product information learned within training folds. '
        f'Five-fold development CV RMSE: {scores[winner]:.2f}; second-seed diagnostic: {stability[winner]:.2f}. '
        'Refit on all labelled data. No external data used.')
    (OUT / 'submission_description.txt').write_text(description, encoding='utf-8')
    print('Submission description:', description)
else:
    print('No supported replacement: keep previous submissions. No new CSV created.')
(OUT / 'metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
